<a href="https://colab.research.google.com/github/omarelmalak/csc490-group2/blob/main/MatchProfile_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 比赛自适应 Pipeline（Match Profile）

给定「哪场比赛」，自动获取赞助商名单和 logo，然后把 Doctor 的判断空间**收窄到这几个品牌**。

## 为什么不靠"多训几个 epoch"

| 做法 | 成本 | 收益 | 风险 |
|---|---|---|---|
| **Doctor 输出掩码**（只在这几个类上 softmax） | **0** | 消灭跨场误标（`new era`/`Armani` 这类直接不可能出现） | 无 |
| **DINO prompt 换成赞助商名单** | 0 | 能认 Doctor 词表外的品牌（如 EA SPORTS） | 词表太长会稀释分数 |
| **原型分类器**（Doctor embedding + 抓到的 logo） | 秒级 | 每类只需 1~5 张图；能加词表外品牌 | 白底 logo 与球衣域有 gap（本 notebook 用合成桥接） |
| Doctor head 掩码微调 | 几分钟 | 略有提升 | 必须用掩码损失，否则洗掉 4890 类知识 |
| **微调 logo detector** | 几十分钟 | ≈0 | **会收窄泛化**（nc=1 只学"像不像 logo"，名单对它无信息量） |

本 notebook 实现前三个（含掩码微调的可选开关）。

## 流程

```text
比赛名（球队列表）
  ├─① Wikipedia infobox 抓 kit_manufacturer / shirt_sponsor / sleeve_sponsor
  │     + 人工补充（抓不到的，如"转播角标"品牌）
  ├─② 每个品牌抓 logo 图（Wikipedia summary → 失败则搜索兜底 → 可人工覆盖）
  │     去白底 / 裁边 / 存 PNG / 拼接触表供你目视确认
  ├─③ 加载 detector + doctor
  │     · doctor 掩码 softmax（只允许赞助商类 + other）
  │     · embedding 原型（logo 贴到本场帧上做"域桥接"，缩小白底→球衣 gap）
  ├─④ 决策：doctor(掩码) → 原型余弦 → DINO(名单 prompt) → 待人工
  │     归属判定用「面积含率 ≥0.55」（能挡掉转播角标）
  ├─⑤ 三个关键帧验证 + 整段聚合，按 role 分表（球衣/装备商/转播叠加）
  └─⑥ 标注图与 CSV 存到 Drive
```

## 你要改的只有第 1 节的 `CLUBS` / `EXTRA_SPONSORS`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================================
# 1. 环境 + 比赛配置
# ============================================================================
import os, re, sys, json, subprocess, glob, shutil, random, time
from pathlib import Path
from urllib.parse import quote
import numpy as np, cv2, torch

IN_COLAB = Path("/content").exists()
if IN_COLAB and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("[drive]", e)

for pkg in ("ultralytics", "transformers", "timm", "pytesseract", "requests", "lxml"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
if not Path("/usr/bin/tesseract").exists():
    subprocess.run(["apt-get", "-qq", "-y", "install", "tesseract-ocr"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE = Path("/content/drive/MyDrive")
W_LOGO = DRIVE / "runs/logo_detector/weights/best.pt"
W_PLAYER = DRIVE / "runs/player_detector/weights/best.pt"
W_DOCTOR = DRIVE / "runs/resnet50_doctor_v2.pth"
VIDEO = DRIVE / "match_0-90s_720p.mp4"

WORK = ROOT / "match_profile"
LOGODIR = WORK / "logos"
for d in (WORK, LOGODIR, WORK / "viz"):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

# ================== 只改这里 ==================
CLUBS = ["Atlético Madrid", "Real Madrid CF"]   # Wikipedia 页面名
MATCH_SEASON = "2023-24"        # !! 必填：赞助商表按赛季，取错赛季会拿到别的赞助商
                                #    写法随意（2023–24 / 2023-24 都行，内部会规范化）
EXTRA_SPONSORS = [                              # 抓不到 / 需要人工补的
    {"brand": "ea sports", "role": "overlay"},
]
LOGO_OVERRIDES = {                              # 抓错图时手动指定本地文件
    # "riyadh air": "/content/drive/MyDrive/ra_logo.png",
}
# ==============================================

# role 决定后续统计口径
ROLE_OF_FIELD = {"kit_manufacturer": "kit", "shirt_sponsor": "shirt",
                 "sleeve_sponsor": "sleeve"}
print("比赛:", " vs ".join(CLUBS))
print("工作目录:", WORK)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
cuda: True Tesla T4
比赛: Atlético Madrid vs Real Madrid CF
工作目录: /content/match_profile


In [ ]:
# ============================================================================
# 2. ① 自动获取赞助商名单
#
# 实测结论（重要）：
#   · 赞助商**不在 infobox**，而在 "Kit suppliers and shirt sponsors" 那一节的 wikitable 里
#   · 表格列名各俱乐部不统一，且大量用 rowspan 表示"连续几个赛季没变"
#   · 因此：手写 wikitext 解析会错位；**必须用 pandas.read_html 解析渲染后的 HTML**
#     （它能正确展开 rowspan，实测 Real Madrid 一次就对：Adidas / Emirates / HP）
#   · 部分页面 HTML 有 rowspan="2;" 这类脏值，read_html 会崩 -> 先做 sanitize
#   · 然而仍有俱乐部表格结构特殊（如 Atlético 表头错位），所以本步只产出**提案**，
#     printed 出来由你确认；`MANUAL_SPONSORS` 永远优先。
# ============================================================================
import io, re, time
import requests
import pandas as pd

UA = {"User-Agent": "Mozilla/5.0 (research; match-profile-pipeline)"}
SESSION = requests.Session()
SESSION.headers.update(UA)
API = "https://en.wikipedia.org/w/api.php"

# 解析不出来 / 想直接指定时，填这里（会覆盖自动结果）
MANUAL_SPONSORS = [
    # {"brand": "adidas",     "role": "kit"},
    # {"brand": "nike",       "role": "kit"},
    # {"brand": "emirates",   "role": "shirt"},
    # {"brand": "riyadh air", "role": "shirt"},
    # {"brand": "mahou",      "role": "sleeve"},
    # {"brand": "ea sports",  "role": "overlay"},
]

# 顺序很重要：具体列名（back/sleeve/front/shorts）必须排在泛化的 sponsor 之前，
# 否则 "Shirt Sponsor (Sleeve)" 会被错判成 shirt（实测踩过）
ROLE_BY_HEADER = [
    (r"kit\s*(manufacturer|supplier)", "kit"),
    (r"\(?\s*back\s*\)?|\bback\b", "shirt_back"),
    (r"\(?\s*sleeve\s*\)?|sleeve\s*sponsor", "sleeve"),
    (r"\(?\s*front\s*\)?|\bfront\b", "shirt"),
    (r"shorts?\s*sponsor|\bshorts\b", "shorts"),
    (r"^shirt\s*sponsor$", "shirt"),
    (r"sponsor", "shirt"),
]


def api(params, tries=4):
    for k in range(tries):
        try:
            r = SESSION.get(API, params=params, timeout=40)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 + 2 * k)
                continue
            return {}
        except Exception:
            time.sleep(1.5 + k)
    return {}


def page_html(page):
    j = api({"action": "parse", "page": page, "prop": "text", "format": "json",
             "formatversion": 2, "redirects": 1})
    if "error" in j:
        print("    [wiki] %s: %s" % (page, j["error"].get("info", "")[:80]))
        return ""
    return j.get("parse", {}).get("text", "") or ""


def sanitize_html(html):
    """rowspan="2;" 这类脏值会让 pandas 崩。"""
    return re.sub(r'(rowspan|colspan)\s*=\s*"(\d+)\s*;?\s*"', r'\1="\2"', html, flags=re.I)


def strip_footnotes(s):
    s = re.sub(r"\[\s*\d+\s*\]", "", str(s))
    s = re.sub(r"\s+", " ", s)
    return s.strip(" ,;")


def role_of(colname):
    c = str(colname).lower()
    for pat, role in ROLE_BY_HEADER:
        if re.search(pat, c):
            return role
    return None


def normalize_season(s):
    s = str(s).lower()
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = re.sub(r"\s+", "", s)
    s = s.replace("present", "").replace("current", "")
    return s


def season_bounds(cell):
    """'2013-2024' / '2023-24' / '2025-' / '2024' -> (start, end)；解析不了返回 (None, None)。"""
    s = normalize_season(cell)
    m = re.match(r"^(\d{4})[-/](\d{4})$", s)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.match(r"^(\d{4})[-/](\d{2})$", s)
    if m:
        return int(m.group(1)), int(m.group(1)[:2] + m.group(2))
    m = re.match(r"^(\d{4})-$", s)
    if m:
        return int(m.group(1)), 9999
    m = re.match(r"^(\d{4})$", s)
    if m:
        return int(m.group(1)), int(m.group(1)) + 1
    return None, None


def pick_row(t, season):
    """赞助商表是按赛季的。这里要处理区间写法：'2013–2024' 覆盖 2023-24。"""
    pcol = next((c for c in t.columns
                 if re.search(r"period|season|年份|赛季", str(c).lower())), None)
    if pcol is None or not season:
        return t.iloc[-1], "最后一行"
    want_start, want_end = season_bounds(season)
    if want_start is None:
        return t.iloc[-1], "赛季写法无法解析: %r" % season
    for _, r in t[::-1].iterrows():
        a, b = season_bounds(r[pcol])
        if a is not None and a <= want_start < b:
            return r, "赛季 %s（表内记作 %s）" % (season, strip_footnotes(r[pcol]))
    return t.iloc[-1], "最后一行（表内没有覆盖 %s 的行）" % season


def club_sponsor_proposal(page, n_show=3):
    """返回 (proposal: list[(brand, role)], 展示用行)。"""
    html = page_html(page)
    if not html:
        return [], []
    tries = [("原样", html), ("清洗后", sanitize_html(html))]
    for tag, h in tries:
        try:
            tables = pd.read_html(io.StringIO(h))
        except Exception:
            continue
        for t in tables:
            roles = {c: role_of(c) for c in t.columns}
            if not any(roles.values()):
                continue
            if len(t) < 2:
                continue
            print("    [%s] 命中表格：%d 行 | 列=%s"
                  % (tag, len(t), [str(c)[:18] for c in t.columns]))
            row, why = pick_row(t, MATCH_SEASON)
            print("    选用: %s" % why)
            rows = [{str(c): strip_footnotes(v) for c, v in r.items()}
                    for _, r in t.tail(n_show).iterrows()]
            prop = []
            for c, v in row.items():
                role = roles.get(c)
                brand = strip_footnotes(v)
                if role and brand and brand.lower() not in ("nan", "none", "") \
                        and not re.fullmatch(r"[\d\W]+", brand):
                    prop.append((brand, role))
            return prop, rows
        break
    return [], []


SPONSORS = {}


def add_sponsor(brand, role, source):
    b = " ".join(str(brand).split()).strip()
    if not b or len(b) > 40:
        return
    for part in re.split(r"\s*(?:/|&|,|;|\band\b)\s*", b):
        part = part.strip()
        if len(part) < 2:
            continue
        key = part.lower()
        if key in SPONSORS:
            continue
        SPONSORS[key] = {"brand": part, "role": role, "source": source}


print("== ① 自动抓取赞助商（Wikipedia 表格，产出提案）==")
for page in CLUBS:
    print("\n  %s" % page)
    prop, rows = club_sponsor_proposal(page)
    if not prop:
        print("    没解析出（改用 MANUAL_SPONSORS）")
    for i, r in enumerate(rows):
        print("     第%d行: %s" % (len(rows) - i, r))
    for brand, role in prop:
        print("     -> 提案: %-20s role=%-10s (wiki)" % (brand, role))
        add_sponsor(brand, role, "wiki:" + page)

for extra in EXTRA_SPONSORS:
    add_sponsor(extra["brand"], extra.get("role", "other"), "manual")
for manual in MANUAL_SPONSORS:          # 人工指定覆盖
    SPONSORS.pop(manual["brand"].lower(), None)
    add_sponsor(manual["brand"], manual.get("role", "other"), "manual")

print("\n== 最终赞助商名单 (%d) ==" % len(SPONSORS))
for k, v in SPONSORS.items():
    print("  %-20s role=%-10s from=%s" % (v["brand"], v["role"], v["source"]))
assert SPONSORS, "一个赞助商都没拿到，填 MANUAL_SPONSORS"
print("\n>>> 请核对上面的名单；不对就填进 MANUAL_SPONSORS 重跑本节")


== ① 自动抓取赞助商（Wikipedia 表格，产出提案）==

  Atlético Madrid
    [清洗后] 命中表格：32 行 | 列=['Period', 'Kit manufacturer', 'Shirt Sponsor (Fro', 'Shirt Sponsor (Bac', 'Shirt Sponsor (Sle', 'Shorts Sponsor']
    选用: 赛季 2023-24（表内记作 2023–24）
     第3行: {'Period': '2024–25', 'Kit manufacturer': 'Nike', 'Shirt Sponsor (Front)': 'Riyadh Air', 'Shirt Sponsor (Back)': 'Hyundai', 'Shirt Sponsor (Sleeve)': 'Kraken', 'Shorts Sponsor': 'ComAve'}
     第2行: {'Period': '2025–', 'Kit manufacturer': 'Nike', 'Shirt Sponsor (Front)': 'Riyadh Air', 'Shirt Sponsor (Back)': 'Visit Rwanda', 'Shirt Sponsor (Sleeve)': 'Kraken', 'Shorts Sponsor': 'ComAve'}
     第1行: {'Period': 'Nike', 'Kit manufacturer': 'nan', 'Shirt Sponsor (Front)': 'nan', 'Shirt Sponsor (Back)': 'nan', 'Shirt Sponsor (Sleeve)': 'nan', 'Shorts Sponsor': 'nan'}
     -> 提案: Nike                 role=kit        (wiki)
     -> 提案: Riyadh Air           role=shirt      (wiki)
     -> 提案: Ria Money Transfer   role=shirt_back (wiki)
     -> 提案: Hyundai            

In [ ]:
# ============================================================================
# 3. ② 抓每个品牌的 logo
#
# 实测结论（重要）：
#   · Wikipedia REST 的 page/summary 接口**会漏掉非自由版权图片**，
#     所以 Emirates / Mahou / EA Sports 用它全都抓不到
#   · 改成解析页面 HTML 里 infobox 的第一个 <img>（走 upload/thumb.wikimedia.org）
#     -> 实测 6/6 全成功（都是 250px 带 alpha 的干净 Logo）
#   · 品牌名到页面标题需要映射：nike -> "Nike, Inc."、emirates -> "Emirates (airline)"、
#     mahou -> "Mahou-San Miguel"
#   · 必须限速（每次 1.2s + 429 重试），否则很快被限流成 0/6
# ============================================================================
import re, time
import numpy as np, cv2

PAGE_TITLES = {          # 名字 -> Wikipedia 页面标题（抓不到时自动搜索兜底）
    "adidas": "Adidas", "nike": "Nike, Inc.", "emirates": "Emirates (airline)",
    "fly emirates": "Emirates (airline)", "riyadh air": "Riyadh Air",
    "mahou": "Mahou-San Miguel", "ea sports": "EA Sports", "ea fc": "EA Sports",
    "electronic arts": "Electronic Arts", "spotify": "Spotify", "puma": "Puma (brand)",
    "hyundai": "Hyundai Motor Company", "barclays": "Barclays",
    "qatar airways": "Qatar Airways", "etihad": "Etihad Airways",
    "mastercard": "Mastercard", "visa": "Visa Inc.", "coca cola": "Coca-Cola",
    "samsung": "Samsung Electronics", "sony": "Sony", "playstation": "PlayStation",
    "xbox": "Xbox", "asus": "Asus", "armani": "Giorgio Armani",
    "under armour": "Under Armour", "new era": "New Era Cap Company",
    "espn": "ESPN", "laliga": "La Liga", "uefa": "UEFA",
}


def infobox_img(page):
    html = page_html(page)
    if not html:
        return None
    m = re.search(r'<table[^>]*class="[^"]*infobox[^"]*"[^>]*>(.*?)</table>',
                  html, flags=re.S | re.I)
    if not m:
        return None
    imgs = re.findall(r'<img[^>]+src="([^"]+)"', m.group(1))
    if not imgs:
        return None
    u = imgs[0]
    return ("https:" + u) if u.startswith("//") else u


def search_titles(q, limit=5):
    j = api({"action": "query", "list": "search", "srsearch": q,
             "format": "json", "srlimit": limit})
    return [h["title"] for h in j.get("query", {}).get("search", [])]


def resolve_logo(brand):
    """返回 (img_url, page_title)。先查映射表，再搜索兜底。"""
    cands = []
    if brand.lower() in PAGE_TITLES:
        cands.append(PAGE_TITLES[brand.lower()])
    cands += [brand, brand + " (company)", brand + " (airline)"]
    seen = set()
    for c in cands:
        if not c or c.lower() in seen:
            continue
        seen.add(c.lower())
        u = infobox_img(c)
        time.sleep(1.2)
        if u:
            return u, c
    for t in search_titles(brand + " logo"):
        if t.lower() in seen:
            continue
        seen.add(t.lower())
        u = infobox_img(t)
        time.sleep(1.2)
        if u:
            return u, t
    return None, None


def strip_white(img_bgr):
    """白底 -> 透明底，裁到内容包围盒。"""
    if img_bgr is None:
        return None
    if img_bgr.ndim == 2:
        img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
    alpha = None
    if img_bgr.shape[2] == 4:
        b, g, r, a = cv2.split(img_bgr)
        rgb = cv2.merge([b, g, r])
        if float(a.min()) < 250:
            alpha = a
    else:
        rgb = img_bgr
    h, w = rgb.shape[:2]
    if max(h, w) > 600:
        s = 600 / max(h, w)
        rgb = cv2.resize(rgb, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
        if alpha is not None:
            alpha = cv2.resize(alpha, (rgb.shape[1], rgb.shape[0]),
                               interpolation=cv2.INTER_AREA)
    if alpha is None:
        whiteness = rgb.min(axis=2).astype(np.float32)
        alpha = np.clip((250.0 - whiteness) / 250.0 * 255.0, 0, 255).astype(np.uint8)
        if alpha.mean() < 5:
            g = cv2.cvtColor(rgb, cv2.COLOR_BGR2GRAY)
            alpha = np.clip(255 - g, 0, 255).astype(np.uint8)
    ys, xs = np.where(alpha > 24)
    if len(xs) < 20:
        return None
    x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
    b, g, r = cv2.split(rgb[y1:y2 + 1, x1:x2 + 1])
    return cv2.merge([b, g, r, alpha[y1:y2 + 1, x1:x2 + 1]])


FETCHED = {}
print("== ② 抓 logo ==")
for key, info in SPONSORS.items():
    brand = info["brand"]
    safe = re.sub(r"[^A-Za-z0-9]+", "_", brand).strip("_").lower()
    out_png = LOGODIR / (safe + ".png")

    override = next((v for k, v in LOGO_OVERRIDES.items() if k.lower() == brand.lower()), None)
    if override and Path(override).exists():
        img = cv2.imread(override, cv2.IMREAD_UNCHANGED)
        origin = "override"
    elif out_png.exists():
        FETCHED[key] = out_png
        print("  %-18s 缓存" % brand)
        continue
    else:
        url, title = resolve_logo(brand)
        if not url:
            print("  %-18s 抓不到（填 LOGO_OVERRIDES 手动指定）" % brand)
            continue
        try:
            r = SESSION.get(url, timeout=60)
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_UNCHANGED)
        except Exception as e:
            print("  %-18s 下载失败 %s" % (brand, str(e)[:60]))
            continue
        origin = "wiki:%s" % title

    rgba = strip_white(img)
    if rgba is None:
        print("  %-18s 去底失败" % brand)
        continue
    cv2.imwrite(str(out_png), rgba)
    FETCHED[key] = out_png
    print("  %-18s %-16s %s" % (brand, "%dx%d" % (rgba.shape[1], rgba.shape[0]), origin))

print("\n抓到 %d / %d" % (len(FETCHED), len(SPONSORS)))

# ---- 接触表：一张图确认全部抓对没有 ----
if FETCHED:
    TH, PAD = 110, 12
    tiles = []
    for key, p in FETCHED.items():
        a = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
        if a is None:
            continue
        s = TH / max(1, a.shape[0])
        a = cv2.resize(a, (max(1, int(a.shape[1] * s)), TH), interpolation=cv2.INTER_AREA)
        tiles.append((SPONSORS[key]["brand"], a))
    if tiles:
        W = sum(t[1].shape[1] for t in tiles) + PAD * (len(tiles) + 1)
        sheet = np.full((TH + 34 + PAD * 2, W, 3), 245, np.uint8)
        x = PAD
        for name, a in tiles:
            h, w = a.shape[:2]
            if a.shape[2] == 4:
                al = a[..., 3:4].astype(np.float32) / 255.0
                patch = (a[..., :3].astype(np.float32) * al + 245 * (1 - al)).astype(np.uint8)
            else:
                patch = a
            sheet[PAD:PAD + h, x:x + w] = patch
            cv2.putText(sheet, name[:16], (x, PAD + TH + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1)
            x += w + PAD
        sheet_path = WORK / "viz" / "sponsor_logos_sheet.png"
        sheet_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(sheet_path), sheet)
        print("接触表 ->", sheet_path, "  ← 打开确认每个品牌抓对了")
        print("   抓错就写 LOGO_OVERRIDES = {'品牌': '本地png'} 后重跑本节")


== ② 抓 logo ==
  Nike               249x89           wiki:Nike, Inc.
  Riyadh Air         250x87           wiki:Riyadh Air
  Ria Money Transfer 249x84           wiki:Ria Money Transfer
  Hyundai            250x33           wiki:Hyundai Motor Company
  Adidas             250x149          wiki:Adidas
  Emirates           250x173          wiki:Emirates (airline)
  ea sports          250x250          wiki:EA Sports

抓到 7 / 7
接触表 -> /content/match_profile/viz/sponsor_logos_sheet.png   ← 打开确认每个品牌抓对了
   抓错就写 LOGO_OVERRIDES = {'品牌': '本地png'} 后重跑本节


In [ ]:
# ============================================================================
# 4. ③ 加载模型 + Doctor 掩码 + embedding 原型（域桥接）
# ============================================================================
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import resnet50
from ultralytics import YOLO

for p in (W_LOGO, W_PLAYER, W_DOCTOR, VIDEO):
    assert Path(p).exists(), "缺少 %s（Drive 没挂载或路径不对）" % p

logo_det = YOLO(str(W_LOGO))
play_det = YOLO(str(W_PLAYER))
PH = {"joueur", "gardien", "player", "goalkeeper", "keeper", "person"}
PIDS = [i for i, n in play_det.names.items() if str(n).strip().lower() in PH] or None

ckpt = torch.load(str(W_DOCTOR), map_location="cpu", weights_only=False)
idx2c = {int(k): v for k, v in ckpt["idx_to_class"].items()}
c2i = {str(v).strip().lower(): int(k) for k, v in idx2c.items()}
TEMP = float(ckpt.get("temperature", 1.0))
OTHER_IDX = ckpt.get("class_to_idx", {}).get(ckpt.get("other_key") or "", None)
if OTHER_IDX is None:
    OTHER_IDX = next((i for i, n in idx2c.items()
                      if re.sub(r"[^a-z0-9]", "", str(n).lower()) == "other"), None)

doctor = resnet50()
doctor.fc = nn.Linear(doctor.fc.in_features, len(idx2c))
doctor.load_state_dict(ckpt["model_state"])
doctor = doctor.to(DEVICE).eval()
# embedding 网络（去掉 fc）
doctor_bb = nn.Sequential(*list(doctor.children())[:-1]).to(DEVICE).eval()
print("doctor %d 类 | T=%.3f | other=%d" % (len(idx2c), TEMP, OTHER_IDX))

# ---------- 赞助商 -> doctor 词表映射 ----------
def norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())


SPONSOR_IDX = {}       # brand -> doctor idx
UNMAPPED = []          # 词表里没有的品牌（靠原型 / DINO）
for key, info in SPONSORS.items():
    b = info["brand"]
    if b.lower() in c2i:
        SPONSOR_IDX[b.lower()] = c2i[b.lower()]
        continue
    # 模糊匹配（"riyadh air" vs "RiyadhAir"）
    hit = next((i for i, n in idx2c.items() if norm(n) == norm(b)), None)
    if hit is None:
        hit = next((i for i, n in idx2c.items()
                    if norm(b) and norm(b) in norm(n) and len(norm(b)) >= 4), None)
    if hit is not None:
        SPONSOR_IDX[b.lower()] = hit
        print("  模糊匹配 %-18s -> %s" % (b, idx2c[hit]))
    else:
        UNMAPPED.append(b)

ALLOW = sorted(set(list(SPONSOR_IDX.values()) + ([OTHER_IDX] if OTHER_IDX is not None else [])))
print("\n可遮蔽类 %d 个 | 词表外品牌: %s" % (len(ALLOW), UNMAPPED or "无"))
assert ALLOW, "一个赞助商都没映射到 doctor 词表，检查名单"


class KAP:
    def __init__(s, size=224, fill=114):
        s.size, s.fill = size, fill
    def __call__(s, img):
        w, h = img.size
        side = max(w, h)
        c = Image.new("RGB", (side, side), (s.fill,) * 3)
        c.paste(img, ((side - w) // 2, (side - h) // 2))
        return c.resize((s.size, s.size), Image.BILINEAR)


tf_doc = T.Compose([KAP(224), T.ToTensor(),
                    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


@torch.no_grad()
def doctor_logits(crops):
    if not crops:
        return torch.zeros((0, len(idx2c)))
    x = torch.stack([tf_doc(Image.fromarray(c)) for c in crops]).to(DEVICE)
    return doctor(x).float()


@torch.no_grad()
def embed(crops, bs=32):
    outs = []
    for i in range(0, len(crops), bs):
        x = torch.stack([tf_doc(Image.fromarray(c)) for c in crops[i:i + bs]]).to(DEVICE)
        outs.append(F.normalize(doctor_bb(x).flatten(1), dim=1))
    return torch.cat(outs) if outs else torch.zeros((0, 2048), device=DEVICE)


def masked_probs(logits):
    """只在 [赞助商类 + other] 上做 softmax，其它 4890 类被先验排除。"""
    sub = logits[:, ALLOW] / TEMP
    return F.softmax(sub, dim=1)


ALLOW_NAMES = [idx2c[i] for i in ALLOW]


def masked_topk(logits, k=3):
    p = masked_probs(logits)
    tk = torch.topk(p, min(k, p.size(1)), dim=1)
    return [[(ALLOW_NAMES[j], float(v)) for v, j in zip(vals, ids)]
            for vals, ids in zip(tk.values.cpu(), tk.indices.cpu())], p.cpu()


# ---------- 原型：把抓到的 logo 贴到本场帧上做「域桥接」 ----------
cap = cv2.VideoCapture(str(VIDEO))
NV = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FRAMES = []
rng = random.Random(0)
for k in range(24):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k * (NV - 2) / 24))
    ok, fr = cap.read()
    if ok:
        FRAMES.append(fr)
cap.release()
print("域桥接用帧:", len(FRAMES))


def paste_on_frame(frame, rgba, scale=0.18):
    h, w = frame.shape[:2]
    a = rgba if rgba is not None else None
    if a is None:
        return None
    rgb = a[..., :3].astype(np.float32)
    alpha = (a[..., 3].astype(np.float32) / 255.0)[..., None]
    tw = int(scale * w)
    th = max(8, int(tw * a.shape[0] / max(1, a.shape[1])))
    rgb = cv2.resize(rgb, (tw, th), interpolation=cv2.INTER_AREA)
    alpha = cv2.resize(alpha[..., 0], (tw, th), interpolation=cv2.INTER_AREA)[..., None]
    px = rng.randint(0, max(0, w - tw))
    py = rng.randint(int(0.1 * h), max(int(0.1 * h) + 1, int(0.8 * h) - th))
    roi = frame[py:py + th, px:px + tw].astype(np.float32)
    if roi.shape[:2] != (th, tw):
        return None
    frame[py:py + th, px:px + tw] = (rgb * alpha + roi * (1 - alpha)).astype(np.uint8)
    return (px, py, px + tw, py + th)


PROTO = {}     # brand -> (2048,) tensor
print("\n== 建原型（logo 原图 + 贴在比赛帧上的合成图）==")
for key, png in FETCHED.items():
    brand = SPONSORS[key]["brand"]
    rgba = cv2.imread(str(png), cv2.IMREAD_UNCHANGED)
    if rgba is None:
        continue
    crops = []
    a = rgba.copy()
    if a.shape[2] == 3:
        a = cv2.cvtColor(a, cv2.COLOR_BGR2RGBA)
    crops.append(cv2.cvtColor(a[..., :3], cv2.COLOR_BGR2RGB))
    for fr in FRAMES[:8]:
        f2 = fr.copy()
        bb = paste_on_frame(f2, a, scale=rng.uniform(0.06, 0.22))
        if bb:
            x1, y1, x2, y2 = bb
            crops.append(cv2.cvtColor(f2[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    if not crops:
        continue
    e = embed(crops).mean(dim=0)
    PROTO[brand.lower()] = F.normalize(e, dim=0)
    print("  %-20s 原型来自 %d 张（含 %d 张合成）" % (brand, len(crops), len(crops) - 1))

PROTO_NAMES = list(PROTO.keys())
if PROTO_NAMES:
    PROTO_M = torch.stack([PROTO[k] for k in PROTO_NAMES])
    print("原型矩阵:", tuple(PROTO_M.shape))


doctor 4903 类 | T=0.707 | other=4902

可遮蔽类 5 个 | 词表外品牌: ['Riyadh Air', 'Ria Money Transfer', 'ea sports']
域桥接用帧: 24

== 建原型（logo 原图 + 贴在比赛帧上的合成图）==
  Nike                 原型来自 9 张（含 8 张合成）
  Riyadh Air           原型来自 9 张（含 8 张合成）
  Ria Money Transfer   原型来自 9 张（含 8 张合成）
  Hyundai              原型来自 9 张（含 8 张合成）
  Adidas               原型来自 9 张（含 8 张合成）
  Emirates             原型来自 9 张（含 8 张合成）
  ea sports            原型来自 9 张（含 8 张合成）
原型矩阵: (7, 2048)


In [ ]:
# ============================================================================
# 5. ④ 归属判定 + 决策（doctor 掩码 → 原型 → DINO → 待人工）
# ============================================================================
import torch.nn.functional as F

PAD, AREA_THR = 0.10, 0.55          # 归属：面积含率（能挡住转播角标）
PROTO_THR = 0.42                    # 原型余弦距离阈值（越小越严）
DOC_THR, DOC_MARGIN = 0.45, 0.15
LN_THR = 0.50
DINO_NAME_THR, DINO_NAME_MARGIN = 0.28, 0.03


def rect(b):
    x1, y1, x2, y2 = b
    return np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], np.float32)


def expand(poly, pad=PAD):
    x1, y1 = poly.min(axis=0); x2, y2 = poly.max(axis=0)
    w, h = max(x2 - x1, 1e-6), max(y2 - y1, 1e-6)
    return np.array([[x1-pad*w, y1-pad*h], [x2+pad*w, y1-pad*h],
                     [x2+pad*w, y2+pad*h], [x1-pad*w, y2+pad*h]], np.float32)


def containment(inner, outer):
    inner = np.asarray(inner, np.float32).reshape(-1, 2)
    outer = np.asarray(outer, np.float32).reshape(-1, 2)
    a = abs(cv2.contourArea(inner))
    if a <= 1e-6:
        return 0.0
    inter, _ = cv2.intersectConvexConvex(inner, outer)
    return float(min(1.0, max(0.0, inter / a)))


def belong(logo_box, player_boxes):
    """返回 (是否属于球衣, 最大含率)。面积含率 >= AREA_THR 才算。"""
    lb = rect(logo_box)
    best = max([containment(lb, expand(rect(p))) for p in player_boxes], default=0.0)
    return best >= AREA_THR, best


def region_of(box, W, H):
    """按位置粗分：顶栏/底栏 = 转播叠加层，其余算场内。"""
    x1, y1, x2, y2 = box
    hh = (y2 - y1) / max(H, 1)
    if y2 <= 0.14 * H and hh < 0.12:
        return "overlay_top"
    if y1 >= 0.86 * H and hh < 0.12:
        return "overlay_bottom"
    return "field"


def crop_with(rgb, box, ctx=0.18):
    h, w = rgb.shape[:2]
    x1, y1, x2, y2 = box
    dx, dy = (x2 - x1) * ctx, (y2 - y1) * ctx
    return rgb[max(0, int(y1-dy)):min(h, int(y2+dy)),
               max(0, int(x1-dx)):min(w, int(x2+dx))]


def decide(rgb, box, crop, logits):
    """核心决策。返回 dict(brand, role, source, conf, detail)"""
    probs = masked_probs(logits)[0]
    order = torch.argsort(probs, descending=True)
    top1 = int(order[0]); top2 = int(order[1]) if probs.numel() > 1 else top1
    brand_allow = ALLOW_NAMES[top1]
    conf = float(probs[top1]); margin = conf - float(probs[top2])
    is_other = (OTHER_IDX is not None and ALLOW[top1] == OTHER_IDX)

    if (not is_other) and conf >= DOC_THR and margin >= DOC_MARGIN:
        role = next((SPONSORS[k]["role"] for k in SPONSORS
                     if norm(SPONSORS[k]["brand"]) == norm(brand_allow)), "?")
        return dict(brand=brand_allow, role=role, source="doctor_masked",
                    conf=conf, detail="conf=%.2f m=%.2f" % (conf, margin))

    # 原型（词表外品牌走这里）
    if PROTO_NAMES:
        e = embed([crop])[0]
        sims = (PROTO_M @ e).cpu().numpy()
        j = int(np.argmax(sims))
        dist = 1.0 - float(sims[j])
        if dist <= PROTO_THR:
            b = PROTO_NAMES[j]
            role = next((SPONSORS[k]["role"] for k in SPONSORS
                         if SPONSORS[k]["brand"].lower() == b), "?")
            return dict(brand=SPONSORS_ROLE(b) or b, role=role, source="proto",
                        conf=1 - dist, detail="cos_dist=%.3f" % dist)

    # DINO 用赞助商名单命名
    names = dino_sponsor_name(rgb, box)
    if names and names[0][1] >= DINO_NAME_THR:
        gap = names[0][1] - (names[1][1] if len(names) > 1 else 0.0)
        if gap >= DINO_NAME_MARGIN:
            b = names[0][0]
            role = next((SPONSORS[k]["role"] for k in SPONSORS
                         if norm(SPONSORS[k]["brand"]) == norm(b)), "?")
            return dict(brand=b, role=role, source="dino_name",
                        conf=names[0][1], detail=str([(n, round(v, 3)) for n, v in names[:3]]))

    return dict(brand="unknown", role="?", source="unknown", conf=conf,
                detail="doctor=%.2f/%.2f proto/ dino 都不达标" % (conf, margin))


def SPONSORS_ROLE(b):
    for k, v in SPONSORS.items():
        if v["brand"].lower() == b:
            return v["brand"]
    return None


# ---------- DINO（沿用之前验证过的写法：带上下文 pad + 只留精确匹配） ----------
_G = {"model": None, "proc": None, "post": None}
SPONSOR_PROMPT_LIST = [v["brand"].lower() for v in SPONSORS.values()]
_SP_SET = set(SPONSOR_PROMPT_LIST)


def load_dino():
    if _G["model"] is not None:
        return True
    try:
        from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
        _G["proc"] = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
        _G["model"] = (AutoModelForZeroShotObjectDetection
                       .from_pretrained("IDEA-Research/grounding-dino-tiny")
                       .to(DEVICE).eval())
        return True
    except Exception as e:
        print("[DINO] 不可用:", e)
        return False


def dino_sponsor_name(rgb, box, pad_f=1.5, box_thr=0.12, text_thr=0.12):
    if not _SP_SET or not load_dino():
        return []
    x1, y1, x2, y2 = [int(v) for v in box]
    H, W = rgb.shape[:2]
    bw, bh = max(1, x2 - x1), max(1, y2 - y1)
    px, py = int(bw * pad_f), int(bh * pad_f)
    view = rgb[max(0, y1-py):min(H, y2+py), max(0, x1-px):min(W, x2+px)]
    if view.size == 0:
        return []
    prompt = " . ".join(SPONSOR_PROMPT_LIST) + " ."
    inp = _G["proc"](images=view, text=prompt, return_tensors="pt").to(DEVICE)
    out = _G["model"](**inp)
    try:
        res = _G["proc"].post_process_grounded_object_detection(
            out, threshold=box_thr, text_threshold=text_thr,
            target_sizes=[view.shape[:2]])[0]
    except TypeError:
        res = _G["proc"].post_process_grounded_object_detection(
            out, inp.get("input_ids"), box_threshold=box_thr,
            text_threshold=text_thr, target_sizes=[view.shape[:2]])[0]
    labels = res.get("text_labels") or res.get("labels") or []
    sc = [float(s) for s in res.get("scores", [])]
    merged = {}
    for l, s in zip(labels, sc):
        k = str(l).strip().lower()
        if k in _SP_SET:
            merged[k] = max(merged.get(k, 0.0), s)
    return sorted(merged.items(), key=lambda kv: -kv[1])[:5]


def analyze(rgb, frame_bgr):
    H, W = rgb.shape[:2]
    pres = play_det(frame_bgr, classes=PIDS, conf=0.25, imgsz=640,
                    verbose=False, device=DEVICE)[0]
    lres = logo_det(frame_bgr, conf=0.10, imgsz=1280, agnostic_nms=True,
                    verbose=False, device=DEVICE)[0]
    pboxes = [tuple(map(float, b)) for b in pres.boxes.xyxy.cpu().numpy()] if len(pres.boxes) else []
    lboxes = [tuple(map(float, b)) for b in lres.boxes.xyxy.cpu().numpy()] if len(lres.boxes) else []

    kept = []
    for b in lboxes:
        ok, cont = belong(b, pboxes)
        if not ok:
            continue
        kept.append((b, cont))
    if not kept:
        return []
    crops = [crop_with(rgb, b) for b, _ in kept]
    logits = doctor_logits(crops)
    out = []
    for i, ((b, cont), crop) in enumerate(zip(kept, crops)):
        d = decide(rgb, b, crop, logits[i:i + 1])
        d.update(box=b, containment=cont, region=region_of(b, W, H))
        out.append(d)
    return out


In [ ]:
# ============================================================================
# 6. ⑤ 三帧验证 + 整段聚合（按 role 分表）+ 存图
# ============================================================================
import subprocess
from collections import defaultdict, Counter
import pandas as pd


def grab_bgr(sec):
    out = "/tmp/mp_%03d.png" % int(sec)
    subprocess.run(["ffmpeg", "-nostdin", "-y", "-loglevel", "error", "-ss", str(sec),
                    "-i", str(VIDEO), "-frames:v", "1", out], check=False)
    return cv2.imread(out)


print("== 三个关键帧 ==")
for sec in (10.0, 30.0, 60.0):
    bgr = grab_bgr(sec)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    recs = analyze(rgb, bgr)
    print("\n[t=%4.1fs] 球衣归属通过的候选 %d 个" % (sec, len(recs)))
    for r in recs:
        print("   %-18s %-8s %-8s %-11s conf=%.2f 含率=%.2f %s   %s"
              % (r["brand"], r["role"], r["region"], r["source"],
                 r["conf"], r["containment"], r["detail"], ""))
    if recs:
        vis = bgr.copy()
        for r in recs:
            x1, y1, x2, y2 = [int(v) for v in r["box"]]
            col = {"doctor_masked": (0, 220, 0), "proto": (230, 0, 230),
                   "dino_name": (0, 200, 255)}.get(r["source"], (150, 150, 150))
            cv2.rectangle(vis, (x1, y1), (x2, y2), col, 3)
            cv2.putText(vis, "%s %.2f (%s)" % (r["brand"], r["conf"], r["source"]),
                        (x1, max(y1 - 6, 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
        p = WORK / "viz" / ("t%03d.png" % int(sec))
        cv2.imwrite(str(p), vis)
        print("   标注图 ->", p)


# ---------- 整段聚合 ----------
MAX_SECONDS = 20          # 先 20 秒试水，确认后再改 90
print("\n== 整段聚合（前 %d 秒，每秒 1 帧）==" % MAX_SECONDS)
cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
limit = int(MAX_SECONDS * fps)
rows = []
i = 0
t0 = time.time()
while i < limit:
    ok, fr = cap.read()
    if not ok:
        break
    if i % int(fps) == 0:
        rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        for r in analyze(rgb, fr):
            rows.append(dict(t_s=round(i / fps, 2), brand=r["brand"], role=r["role"],
                             source=r["source"], conf=round(r["conf"], 3),
                             containment=round(r["containment"], 3), region=r["region"]))
    i += 1
cap.release()
df = pd.DataFrame(rows)
print("评估 %d 帧，命中 %d 条，用时 %.0fs" % (i // int(fps), len(df), time.time() - t0))

if len(df):
    agg = (df.groupby(["role", "brand"]).size().rename("seconds_seen").reset_index()
           .sort_values(["role", "seconds_seen"], ascending=[True, False]))
    print("\n按 role 分表（秒数 ≈ 出现帧数）:")
    print(agg.to_string(index=False))
    print("\n按来源分:", dict(Counter(df["source"])))
    agg.to_csv(WORK / "exposure_by_role.csv", index=False)
    df.to_csv(WORK / "exposure_detail.csv", index=False)
    print("\nCSV ->", WORK / "exposure_by_role.csv")
else:
    print("没有命中：检查 sponser 名单/阈值")


== 三个关键帧 ==

[t=10.0s] 球衣归属通过的候选 1 个
   Adidas             kit      field    doctor_masked conf=1.00 含率=1.00 conf=1.00 m=1.00   
   标注图 -> /content/match_profile/viz/t010.png

[t=30.0s] 球衣归属通过的候选 2 个
   Emirates           shirt    field    doctor_masked conf=1.00 含率=1.00 conf=1.00 m=1.00   
   Adidas             kit      field    doctor_masked conf=1.00 含率=1.00 conf=1.00 m=1.00   
   标注图 -> /content/match_profile/viz/t030.png


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

/tmp/ipykernel_1253/77822416.py:154: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  sc = [float(s) for s in res.get("scores", [])]



[t=60.0s] 球衣归属通过的候选 2 个
   unknown            ?        field    unknown     conf=0.82 含率=1.00 doctor=0.82/0.73 proto/ dino 都不达标   
   unknown            ?        field    unknown     conf=0.82 含率=1.00 doctor=0.82/0.64 proto/ dino 都不达标   
   标注图 -> /content/match_profile/viz/t060.png

== 整段聚合（前 20 秒，每秒 1 帧）==
评估 20 帧，命中 62 条，用时 5s

按 role 分表（秒数 ≈ 出现帧数）:
   role      brand  seconds_seen
      ?    unknown             2
    kit       Nike            27
    kit     Adidas            16
overlay  ea sports             2
  shirt   Emirates             7
  shirt Riyadh Air             5
  shirt riyadh air             1
 sleeve    Hyundai             2

按来源分: {'doctor_masked': 52, 'proto': 6, 'dino_name': 2, 'unknown': 2}

CSV -> /content/match_profile/exposure_by_role.csv
